# APIs de LLM — práctica (Sesión 1)

**OpenAI:** ejercicios **2a** (Responses API, recomendada en el máster) y **2b** (Chat Completions, patrón que comparten muchos proveedores). **Anthropic:** ejercicio **3** (Messages API).

Necesitas Python 3.11+, `uv sync` en el repo y las claves en Colab (secretos) o en variables de entorno. **No subas API keys a Git.**

## OpenAI: dos APIs, una recomendación

- **Responses API** (`client.responses.create`): interfaz actual recomendada por OpenAI para proyectos nuevos (marzo 2025). Unifica capacidades previas, `instructions` + `input`, estado entre turnos y herramientas integradas. **Es la que usa el máster como referencia principal.**
- **Chat Completions** (`client.chat.completions.create`): API anterior, seguirá soportada; el patrón `messages` con roles (`system`, `user`, `assistant`) es el que verás al abstraer proveedores (LiteLLM, etc.).

En Colab: `%pip install -q openai` si hace falta. Secreto `OPENAI_API_KEY` (con acceso al notebook). En local: `export OPENAI_API_KEY=...`.

## Ejercicio 2a — OpenAI Responses API (`responses.create`)

Llamada recomendada: `instructions` equivale al rol de *system*; `input` lleva el texto del usuario (también admite estructuras más ricas). Revisa `output_text` y `usage` en la respuesta.

Documentación: [Responses](https://platform.openai.com/docs/guides/text).

In [ ]:
# %pip install -q openai

import os
from getpass import getpass

from openai import OpenAI


def obtener_api_key_openai() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if key:
        return key
    return getpass("OPENAI_API_KEY: ").strip()


api_key = obtener_api_key_openai()
if not api_key:
    raise ValueError("Falta OPENAI_API_KEY")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"  # TODO: otro modelo compatible con Responses

respuesta = client.responses.create(
    model=MODEL,
    instructions=(
        "Eres un tutor del máster de IA. Responde en español, "
        "máximo 3 frases, sin listas largas."
    ),
    input="Explica qué es un system prompt en una petición a la API.",
    temperature=0.2,
    max_output_tokens=200,
)

print("--- Contenido (output_text) ---")
print(respuesta.output_text)
print("--- Metadatos ---")
print("model:", respuesta.model)
print("id:", respuesta.id)
print("status:", respuesta.status)
if respuesta.usage:
    print("usage:", respuesta.usage.model_dump())

## Ejercicio 2b — OpenAI Chat Completions (`chat.completions.create`)

Misma intención que el 2a, con el patrón **clásico** `messages`: `system` + `user`. Es el estilo que suelen mapear agregadores y otros proveedores.

Documentación: [Chat Completions](https://platform.openai.com/docs/guides/text-generation).

In [ ]:
# %pip install -q openai

import os
from getpass import getpass

from openai import OpenAI


def obtener_api_key_openai() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if key:
        return key
    return getpass("OPENAI_API_KEY: ").strip()


api_key = obtener_api_key_openai()
if not api_key:
    raise ValueError("Falta OPENAI_API_KEY")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

respuesta = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Eres un tutor del máster de IA. Responde en español, "
                "máximo 3 frases, sin listas largas."
            ),
        },
        {
            "role": "user",
            "content": "Explica qué es un system prompt en una petición a la API.",
        },
    ],
    temperature=0.2,
    max_tokens=200,
)

msg = respuesta.choices[0].message.content
print("--- Contenido ---")
print(msg)
print("--- Metadatos ---")
print("model:", respuesta.model)
print("id:", respuesta.id)
if respuesta.usage:
    print("usage:", respuesta.usage.model_dump())

## Ejercicio 3 — Anthropic (Messages API)

Misma pregunta que en OpenAI: **system** (parámetro `system`) + **user** en `messages`, texto agregado y **tokens** (`input_tokens` / `output_tokens`).

- **Colab:** `%pip install -q anthropic` y secreto `ANTHROPIC_API_KEY` (acceso al notebook).
- **Local:** `export ANTHROPIC_API_KEY=...`.

### Si ves `401` / `invalid x-api-key`

- Clave de **[Anthropic Console](https://console.anthropic.com/)** (`sk-ant-...`), no la de OpenAI.
- Nombre del secreto: `ANTHROPIC_API_KEY`, valor sin espacios extra.

In [ ]:
# %pip install -q anthropic

import os
from getpass import getpass

from anthropic import Anthropic


def obtener_api_key_anthropic() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("ANTHROPIC_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
    if key:
        return key
    return getpass("ANTHROPIC_API_KEY: ").strip()


api_key = obtener_api_key_anthropic().strip()
if not api_key:
    raise ValueError("Falta ANTHROPIC_API_KEY")

client = Anthropic(api_key=api_key)
MODEL = "claude-haiku-4-5-20251001"

respuesta = client.messages.create(
    model=MODEL,
    max_tokens=200,
    system=(
        "Eres un tutor del máster de IA. Responde en español, "
        "máximo 3 frases, sin listas largas."
    ),
    messages=[
        {
            "role": "user",
            "content": "Explica qué es un system prompt en una petición a la API.",
        }
    ],
    temperature=0.2,
)

texto = "".join(bloque.text for bloque in respuesta.content if hasattr(bloque, "text"))

print("--- Contenido ---")
print(texto)
print("--- Metadatos ---")
print("model:", respuesta.model)
print("id:", respuesta.id)
print("stop_reason:", respuesta.stop_reason)
if respuesta.usage:
    print(
        "usage:",
        {
            "input_tokens": respuesta.usage.input_tokens,
            "output_tokens": respuesta.usage.output_tokens,
        },
    )